# Day 142 — Streamlit Advanced: Session State, Tabs, Forms & Multi-Page Apps
**Month 8 | Day 2 | Deployment Track**

---
| | |
|---|---|
| **Dataset** | FreelanceHub India — 500 projects, seed=141 |
| **Tools** | streamlit, pandas, plotly, numpy |
| **Max Score** | 80 pts + 10★ bonus |
| **GitHub** | Month8-Streamlit-Portfolio |
| **Output** | `app_day142.py` — a multi-section interactive Streamlit app |

---
> **Rule:** Attempt every task cell before checking Section 4.  
> Write your intent as a comment before writing any code.  
> Session state bugs are the #1 Streamlit interview topic — get this right.


---
## Section 1 — Dataset Reference
> Never modify the raw data. This section is read-only context.

In [6]:
# Section 1: Raw Data — FreelanceHub India (seed=141, 500 rows)
# Run once to confirm your dataset matches the answer key numbers.
import pandas as pd
import numpy as np

np.random.seed(141)
n = 500

categories    = ['Web Development', 'Data Analytics', 'ML/AI', 'Design',
                 'Content Writing', 'Digital Marketing']
cities        = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
levels        = ['Rising Talent', 'Level 1', 'Level 2', 'Top Rated']
statuses      = ['Completed', 'In Progress', 'Cancelled']
months        = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df_raw = pd.DataFrame({
    'project_id'       : [f'P{str(i).zfill(3)}' for i in range(1, n+1)],
    'category'         : np.random.choice(categories, n, p=[0.25,0.20,0.15,0.15,0.15,0.10]),
    'budget_inr'       : np.random.randint(5000, 150001, n),
    'duration_days'    : np.random.randint(3, 91, n),
    'client_rating'    : np.round(np.random.uniform(2.5, 5.0, n), 1),
    'freelancer_level' : np.random.choice(levels, n, p=[0.30,0.30,0.25,0.15]),
    'status'           : np.random.choice(statuses, n, p=[0.65,0.25,0.10]),
    'city'             : np.random.choice(cities, n),
    'platform_fee_pct' : np.random.choice([10, 15, 20], n, p=[0.20,0.50,0.30]),
    'month'            : np.random.choice(months, n)
})
df_raw['net_revenue'] = np.round(df_raw['budget_inr'] * (1 - df_raw['platform_fee_pct']/100), 2)

print(f"Shape : {df_raw.shape}")
print(f"Cols  : {list(df_raw.columns)}")
print(f"\nFirst 3 rows:")
print(df_raw.head(3).to_string(index=False))


2026-05-29 22:38:17.330 No runtime found, using MemoryCacheStorageManager
2026-05-29 22:38:17.335 No runtime found, using MemoryCacheStorageManager
2026-05-29 22:38:17.336 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


---
## Section 2 — Concept Notes

### Why Day 141 wasn't enough

Day 141 built a single-page Streamlit app. Every widget change triggers a **full script rerun** — Streamlit executes your `.py` top to bottom. This creates two real problems:

1. **State loss** — user selections vanish on rerun unless you store them
2. **App complexity** — analytics tools need multiple views, not one scrolling page

Day 142 solves both.

---

### 2.1 — `st.session_state`

```python
# Session state is a dict-like object that PERSISTS across reruns
if 'watchlist' not in st.session_state:
    st.session_state.watchlist = []          # Initialize once

if st.button("Add ML/AI"):
    st.session_state.watchlist.append("ML/AI")   # Persists on next rerun

st.write(st.session_state.watchlist)         # Still there after rerun
```

**Key rules:**
- Always **initialise with an `if key not in` guard** — never directly assign on first run
- Use it for: counters, user preferences, pinned selections, multi-step form data
- `st.session_state` is **per-session** — cleared when the browser tab closes

---

### 2.2 — Multi-Page Apps (simulated within one file)

Real multi-page structure uses a `pages/` folder or `st.navigation` (Streamlit ≥ 1.36).  
For practice and single-file apps, simulate pages with a **sidebar radio/selectbox + conditional rendering**:

```python
page = st.sidebar.radio("Navigate", ["📊 Overview", "📂 Category", "🏙 City"])
if page == "📊 Overview":
    show_overview()
elif page == "📂 Category":
    show_category()
```

This is the pattern used in 90% of freelance deliverables because it ships as one `.py` file.

---

### 2.3 — `st.form` + `st.form_submit_button`

```python
with st.form("calculator"):
    budget = st.number_input("Budget (₹)", min_value=5000, value=80000, step=5000)
    fee    = st.selectbox("Platform fee %", [10, 15, 20])
    days   = st.number_input("Duration (days)", min_value=1, value=30)
    submit = st.form_submit_button("Calculate")

if submit:
    net   = budget * (1 - fee / 100)
    daily = net / days
    st.metric("Net Revenue", f"₹{net:,.0f}")
    st.metric("Daily Rate",  f"₹{daily:,.0f}")
```

**Why it matters:** Without `st.form`, every widget change triggers a rerun. With `st.form`, **nothing runs until Submit is clicked**. Use for: calculators, filters that are expensive to compute, multi-field inputs.

---

### 2.4 — `st.tabs`

```python
tab1, tab2, tab3 = st.tabs(["Overview", "Category", "City"])
with tab1:
    st.metric("Total Projects", 500)
with tab2:
    st.bar_chart(cat_df)
with tab3:
    st.dataframe(city_df)
```

Tabs do not rerun the whole app — they switch the visible panel client-side. Use for: multi-view dashboards, before/after comparisons, drill-down analysis.

---

### 2.5 — `st.expander`

```python
with st.expander("📋 Show raw data table"):
    st.dataframe(df[['project_id','category','budget_inr','status']])
```

Progressive disclosure — hide detail until requested. Use for: raw data, methodology notes, verbose outputs that clutter the main view.

---

### 2.6 — `st.download_button`

```python
csv = df.to_csv(index=False).encode('utf-8')
st.download_button(
    label="📥 Download Completed Projects CSV",
    data=csv,
    file_name="completed_projects.csv",
    mime="text/csv"
)
```

**Critical detail:** `encode('utf-8')` is required. `mime="text/csv"` tells the browser to treat it as a file, not a web response.

---

### 2.7 — Interview framing

*"What's the difference between `st.cache_data` and `st.session_state`?"*

> "`st.cache_data` is for functions — it memoises the return value so a CSV isn't re-read on every rerun. It's shared across all users in the same session.  
> `st.session_state` is for user-specific state — a counter, a pinned selection, a watchlist. It's unique to each user's browser session. Use cache for data loading; use session_state for user interaction."


---
## Section 3 — Practice Tasks (80 pts + 10★ bonus)

Build **one file: `app_day142.py`** that is a complete, runnable Streamlit app.  
Each task adds one feature to that file. Complete in order.

> Write your intent as a plain-English comment above each code block.  
> Run `streamlit run app_day142.py` to test after each task.


### T1 — Session State: Watchlist (10 pts)

**Problem:** User selects categories to track. Selections must persist when other widgets change.

**Spec:**
- If `'watchlist'` not in `st.session_state`, initialise as empty list
- Sidebar `st.multiselect` for category selection; on button click, add to watchlist
- `st.sidebar.write()` to show current watchlist count and contents
- Watchlist persists when budget slider or other widgets change

**Expected output:** Sidebar shows `Watchlist (N items): ['ML/AI', 'Design']` after user adds categories

In [ ]:
# T1 – Session state watchlist
# Intent: store user's category selections in session_state so they survive reruns.

if 'watchlist' not in st.session_state:
    st.session_state.watchlist = []

st.sidebar.title("FreelanceHub India")
st.sidebar.markdown("---")
st.sidebar.subheader("📌 Category Watchlist")
add_cat = st.sidebar.selectbox("Select category to track", categories)
if st.sidebar.button("Add to Watchlist"):
    if add_cat not in st.session_state.watchlist:
        st.session_state.watchlist.append(add_cat)
if st.sidebar.button("Clear Watchlist"):
    st.session_state.watchlist = []
st.sidebar.write(f"**Watchlist ({len(st.session_state.watchlist)} items):**")
st.sidebar.write(st.session_state.watchlist if st.session_state.watchlist else "Empty — add categories above")

### T2 — Multi-Page Navigation (15 pts)

**Problem:** The app needs 3 views: Overview / Category Analysis / City Analysis.

**Spec:**
- `st.sidebar.radio()` with options `['📊 Overview', '📂 Category', '🏙 City']`
- Each page renders different content (metrics / bar chart / table)
- Page selection stored in `st.session_state['page']` so it persists
- `st.title()` at top of each page reflects the current view name

**Expected:** Switching to '📂 Category' shows a bar chart of avg budget by category sorted descending. Switching back to '📊 Overview' does not reset any other session state.

In [ ]:
# T2 – Multi-page navigation with session state
# Intent: simulate multi-page app using sidebar radio + conditional rendering.

if 'page' not in st.session_state:
    st.session_state.page = '📊 Overview'

page = st.sidebar.radio(
    "Navigate",
    ['📊 Overview', '📂 Category', '🏙️ City'],
    index=['📊 Overview','📂 Category','🏙️ City'].index(st.session_state.page)
)
st.session_state.page = page

### T3 — Tabs: Three-View Dashboard (15 pts)

**Problem:** The Overview page needs 3 sub-views without scrolling.

**Spec:**
- `tab_ov, tab_cat, tab_city = st.tabs(['Overview', 'Category', 'City'])`
- **Tab 1 — Overview:** 4 metric cards in `st.columns(4)`: Total Projects (500), Avg Budget (₹79,030), Completion Rate (63.2%), Avg Rating (3.81)
- **Tab 2 — Category:** Plotly bar chart of `avg_budget` by category, sorted descending; subheader 'Category Avg Budget (₹)'
- **Tab 3 — City:** `st.dataframe()` of city-level summary: city, projects, avg_budget, avg_rating; sorted by avg_budget desc

**Answer key values:** Top category = ML/AI at ₹84,800 | Top city = Kolkata at ₹84,266

In [ ]:
if page == '📊 Overview':
    st.title("📊 FreelanceHub India — Overview")

    # T3: Tabs
    tab_ov, tab_cat, tab_city = st.tabs(["Overview", "Category", "City"])

    with tab_ov:
        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Total Projects", f"{len(df):,}")
        col2.metric("Avg Budget", f"₹{df['budget_inr'].mean():,.0f}")
        col3.metric("Completion Rate", f"{(df['status']=='Completed').mean()*100:.1f}%")
        col4.metric("Avg Rating", f"{df['client_rating'].mean():.2f}")

    with tab_cat:
        st.subheader("Category Avg Budget (₹)")
        cat_df = df.groupby('category')['budget_inr'].mean().reset_index()
        cat_df = cat_df.sort_values('budget_inr', ascending=False)
        fig = px.bar(cat_df, x='category', y='budget_inr',
                     color='budget_inr', color_continuous_scale='Blues',
                     labels={'budget_inr':'Avg Budget (₹)'})
        fig.update_layout(showlegend=False, coloraxis_showscale=False)
        st.plotly_chart(fig, use_container_width=True)

    with tab_city:
        st.subheader("City Performance")
        city_df = df.groupby('city').agg(
            Projects=('project_id','count'),
            Avg_Budget=('budget_inr','mean'),
            Avg_Rating=('client_rating','mean')
        ).reset_index().sort_values('Avg_Budget', ascending=False)
        city_df['Avg_Budget'] = city_df['Avg_Budget'].round(0).astype(int)
        city_df['Avg_Rating'] = city_df['Avg_Rating'].round(2)
        st.dataframe(city_df, use_container_width=True, hide_index=True)

### T4 — Form: Profitability Calculator (10 pts)

**Problem:** Freelancers need to calculate net revenue before bidding.

**Spec:**
- `st.form('profit_calc')` with:
  - `st.number_input('Project Budget (₹)', min_value=5000, max_value=150000, value=80000, step=5000)`
  - `st.selectbox('Platform Fee %', [10, 15, 20], index=1)` (default 15%)
  - `st.number_input('Duration (days)', min_value=1, max_value=90, value=30)`
  - `st.form_submit_button('Calculate')`
- On submit: show 3 `st.metric()` cards: Net Revenue, Daily Rate, Hourly Rate (8h day)
- Nothing recalculates until Submit is clicked

**Answer key:** Rs 80k @ 15% = Net ₹68,000 | Daily ₹2,267 | Hourly ₹283

In [ ]:
        st.markdown("---")
        st.subheader("🧮 Project Profitability Calculator")
        with st.form("profit_calc"):
            c1, c2, c3 = st.columns(3)
            budget = c1.number_input("Budget (₹)", min_value=5000, max_value=150000,
                                     value=80000, step=5000)
            fee    = c2.selectbox("Platform Fee %", [10, 15, 20], index=1)
            days   = c3.number_input("Duration (days)", min_value=1, max_value=90, value=30)
            submit = st.form_submit_button("Calculate")

        if submit:
            net   = budget * (1 - fee / 100)
            daily = net / days
            hourly = daily / 8
            m1, m2, m3 = st.columns(3)
            m1.metric("Net Revenue", f"₹{net:,.0f}")
            m2.metric("Daily Rate",  f"₹{daily:,.0f}")
            m3.metric("Hourly Rate", f"₹{hourly:,.0f}")

### T5 — Expander: Raw Data Reveal (10 pts)

**Spec:**
- `st.expander('📋 Show project data (filtered)')` below the metric cards
- Inside: `st.dataframe()` of the filtered dataset showing only:
  `['project_id', 'category', 'budget_inr', 'net_revenue', 'status', 'client_rating', 'city']`
- Row count shown above the table: `st.caption(f'{len(filtered_df)} projects shown')`
- Table is `use_container_width=True`

**Expected:** Expander is collapsed by default. Expanding shows the 7-column table.

In [ ]:
        st.markdown("---")
        with st.expander("📋 Show project data (500 rows)"):
            cols_show = ['project_id','category','budget_inr','net_revenue',
                         'status','client_rating','city']
            st.caption(f"{len(df)} projects shown")
            st.dataframe(df[cols_show], use_container_width=True)

### T6 — Session State: Pinned Comparison (10 pts)

**Problem:** User wants to lock one category as a baseline and compare others against it.

**Spec:**
- `st.selectbox('📌 Pin a baseline category', categories)` — default index 2 (ML/AI)
- Button `'Set as Baseline'` → saves to `st.session_state['baseline']`
- Below the selectbox, show a two-row comparison table:

| Metric | Baseline (ML/AI) | Platform Avg |
|---|---|---|
| Avg Budget (₹) | ₹84,800 | ₹79,030 |
| Completion Rate | 53.5% | 63.2% |
| Avg Rating | 3.84 | 3.81 |

- Comparison updates only when 'Set as Baseline' is clicked (uses session_state)

**Expected:** Changing the selectbox without clicking the button does NOT update the table.

In [ ]:
        # T6: Pinned comparison
        if 'baseline' not in st.session_state:
            st.session_state.baseline = 'ML/AI'

        st.subheader("📌 Baseline Category Comparison")
        sel_baseline = st.selectbox("Select baseline category", categories,
                                     index=categories.index('ML/AI'))
        if st.button("Set as Baseline"):
            st.session_state.baseline = sel_baseline

        baseline = st.session_state.baseline
        b_df   = df[df['category'] == baseline]
        plat_avg_budget      = df['budget_inr'].mean()
        plat_completion      = (df['status'] == 'Completed').mean() * 100
        plat_rating          = df['client_rating'].mean()
        b_avg_budget         = b_df['budget_inr'].mean()
        b_completion         = (b_df['status'] == 'Completed').mean() * 100
        b_rating             = b_df['client_rating'].mean()

        comp_df = pd.DataFrame({
            'Metric'          : ['Avg Budget (₹)', 'Completion Rate (%)', 'Avg Rating'],
            f'Baseline: {baseline}': [f"₹{b_avg_budget:,.0f}", f"{b_completion:.1f}%", f"{b_rating:.2f}"],
            'Platform Avg'    : [f"₹{plat_avg_budget:,.0f}", f"{plat_completion:.1f}%", f"{plat_rating:.2f}"]
        })
        st.dataframe(comp_df, use_container_width=True, hide_index=True)
        st.markdown("---")

### T7 — NRA Insight (10 pts)

**Spec:** Display one `st.info()` block with a full NRA insight.

**Counter-intuitive finding to use:**
- Top Rated avg budget: ₹75,616 — LOWER than Rising Talent at ₹77,853
- Premium gap: −₹2,236 (Top Rated earns less per project)
- Top Rated completion rate: 66.7% vs platform avg 63.2%

**NRA format:**
- **Number:** The specific figures above (both levels, the gap)
- **Reason:** Why higher badge doesn't equal higher budget (niche specialisation, longer project cycles, premium retainer vs one-off work)
- **Action:** Concrete step (e.g., target ML/AI + Design categories at ₹84,800 avg rather than chasing badge level; raise ML/AI bid floor to ₹70,000)

**Insight must reference printed/computed values, not hardcoded text.**

In [ ]:
# T7 – NRA Insight (computed from data, not hardcoded)
# Goal: Show that Top Rated freelancers earn less per project than Rising Talent,
#       then explain why and give actionable steps.

# Compute from data
top_rated_avg = df[df['freelancer_level'] == 'Top Rated']['budget_inr'].mean()
rising_talent_avg = df[df['freelancer_level'] == 'Rising Talent']['budget_inr'].mean()
gap = top_rated_avg - rising_talent_avg
top_rated_completion = (df[df['freelancer_level'] == 'Top Rated']['status'] == 'Completed').mean() * 100
platform_completion = (df['status'] == 'Completed').mean() * 100

st.info(
    f"**📊 NRA – Counter-Intuitive Level Pricing Finding**\n\n"
    f"**Number:** Top Rated freelancers average ₹{top_rated_avg:,.0f}/project – "
    f"₹{abs(gap):,.0f} **LESS** than Rising Talent at ₹{rising_talent_avg:,.0f}. "
    f"Top Rated completion rate: {top_rated_completion:.1f}% vs platform avg {platform_completion:.1f}%.\n\n"
    f"**Reason:** Badge level does not dictate project budget. Top Rated freelancers "
    f"often work on longer, complex retainers (lower per-project ₹, higher lifetime value) "
    f"while Rising Talent competes on price for quick one-off projects. "
    f"Platform‑level aggregation hides this segment behaviour.\n\n"
    f"**Action:** Target ML/AI and Design categories (avg ₹84,800) instead of chasing badge status. "
    f"Set Upwork bid floor at ₹70,000 for ML/AI projects. Update Fiverr profile to highlight "
    f"ML/AI specialisation within 7 days."
)

### ★ Bonus — Download Button (10★)

**Spec:**
- Filter `df` to `status == 'Completed'` (316 rows)
- Convert to CSV with `df.to_csv(index=False).encode('utf-8')`
- `st.download_button(label='📥 Download Completed Projects', data=csv, file_name='completed_projects.csv', mime='text/csv')`
- Add `st.caption()` below showing total net revenue in the export: ₹21,122,581

**Expected:** Clicking the button downloads a 316-row CSV named `completed_projects.csv`.

In [ ]:
        st.markdown("---")
        completed = df[df['status'] == 'Completed']
        csv = completed.to_csv(index=False).encode('utf-8')
        st.download_button(
            label="📥 Download Completed Projects",
            data=csv,
            file_name="completed_projects.csv",
            mime="text/csv"
        )
        st.caption(f"Export contains {len(completed)} projects | "
                   f"Total net revenue: ₹{completed['net_revenue'].sum():,.0f}")
elif page == '📂 Category':
    st.title("📂 Category Deep Dive")
    cat_summary = df.groupby('category').agg(
        Projects=('project_id','count'),
        Avg_Budget=('budget_inr','mean'),
        Avg_Rating=('client_rating','mean'),
        Completion_Rate=('status', lambda x: f"{(x=='Completed').mean()*100:.1f}%")
    ).reset_index().sort_values('Avg_Budget', ascending=False)
    cat_summary['Avg_Budget'] = cat_summary['Avg_Budget'].apply(lambda x: f"₹{x:,.0f}")
    st.dataframe(cat_summary, use_container_width=True, hide_index=True)

elif page == '🏙️ City':
    st.title("🏙️ City Performance")
    city_full = df.groupby('city').agg(
        Projects=('project_id','count'),
        Avg_Budget=('budget_inr','mean'),
        Avg_Rating=('client_rating','mean'),
        Completion_Rate=('status', lambda x: f"{(x=='Completed').mean()*100:.1f}%")
    ).reset_index().sort_values('Avg_Budget', ascending=False)
    city_full['Avg_Budget'] = city_full['Avg_Budget'].apply(lambda x: f"₹{x:,.0f}")
    st.dataframe(city_full, use_container_width=True, hide_index=True)

---
## Section 4 — Scoring Rubric

| Task | Points | What is assessed |
|---|---|---|
| **T1** Session State Watchlist | 10 | `if key not in` guard ✓, button appends correctly ✓, persists on rerun ✓ |
| **T2** Multi-Page Navigation | 15 | `st.sidebar.radio` ✓, page stored in session_state ✓, each page renders distinct content ✓ |
| **T3** Tabs Dashboard | 15 | 3 tabs created ✓, 4 metric cards with correct values ✓, chart sorted desc ✓, city table correct ✓ |
| **T4** Form Calculator | 10 | `st.form` context manager ✓, 3 inputs ✓, submit button ✓, 3 metrics correct ✓ |
| **T5** Expander | 10 | Collapsed by default ✓, 7 columns only ✓, caption with row count ✓ |
| **T6** Pinned Comparison | 10 | Session state updated only on button click ✓, comparison table correct values ✓ |
| **T7** NRA Insight | 10 | Number (both levels + gap) ✓, Reason (explains mechanism) ✓, Action (specific, time-bound) ✓ |
| **★ Bonus** Download Button | 10★ | 316-row filter ✓, encode utf-8 ✓, correct filename ✓, caption with total ₹ ✓ |

**Total: 80 pts + 10★ bonus**

---

### Deduction guide

| Error type | Deduction |
|---|---|
| Missing `if key not in` guard (causes KeyError on second rerun) | −5 |
| Hardcoded metric values instead of computed | −3 per metric |
| Form without `st.form_submit_button` (acts like no form) | −5 |
| NRA action not time-bound or not specific | −3 |
| Comparison table updates on selectbox change (not on button) | −5 |
| Missing `encode('utf-8')` on download | −3 |
| Wrong column count in expander (not 7) | −2 |

---

### Key Takeaway — Day 142

**`st.session_state` is the most misused Streamlit pattern.** The most common bug: assigning directly (`st.session_state.x = []`) without an `if not in` guard — this resets the state on every rerun, making it functionally useless. The guard pattern (`if 'x' not in st.session_state: st.session_state.x = []`) is non-negotiable for any state that needs to accumulate or persist. Get this right once and it works in every Streamlit app you build.

---

### Interview Answer — Day 142

*"What is `st.session_state` and when would you use it vs `st.cache_data`?"*

> "`st.session_state` is a per-user, per-session dictionary that persists across script reruns. Use it when you need to track user actions — a selection counter, a watchlist, a multi-step form's intermediate values. `st.cache_data` is different — it caches the return value of a function so a CSV isn't re-read every rerun, and it's shared across reruns. Use `cache_data` for data loading; use `session_state` for user interaction. Confusing the two is a common Streamlit interview trip-up."

---

### Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ |
| **142** | **Session State + Tabs + Forms** | _/80 + _★ |
